In [19]:
import pandas as pd

df = pd.read_csv("../data/CRMLSSold_merge.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_6212/3104117343.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/CRMLSSold_merge.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,month,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,202401,NaN,NaN,NaN,NaN
1,HighDesert,HighDesert,NaN,NaN,NaN,NaN,NaN,0.0,535486633,eabrown@lee-associates.com,...,NaN,52320.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN
2,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,75000.0,529986282,Joe@9WINWIN.com,...,NaN,217364.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN
3,InlandValleys,InlandValleys,NaN,True,NaN,NaN,NaN,199000.0,529618166,carolthefinder@yahoo.com,...,NaN,217800.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,NaN,NaN,19500.0,522614340,jtavisola@tavisola.com,...,0.0,108883.0,NaN,CRMLS,CRMLS_CRM,202401,NaN,NaN,NaN,NaN


In [20]:
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(exclude=["number"]).columns

print("Numerical columns:", len(num_cols))
print(num_cols)

Numerical columns: 32
Index(['OriginalListPrice', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'FireplacesTotal', 'AboveGradeFinishedArea', 'ListingKeyNumeric',
       'TaxAnnualAmount', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt',
       'StreetNumberNumeric', 'BathroomsTotalInteger', 'TaxYear',
       'BuildingAreaTotal', 'BedroomsTotal', 'ElementarySchoolDistrict',
       'BelowGradeFinishedArea', 'CoveredSpaces', 'Stories', 'LotSizeArea',
       'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee',
       'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict', 'month',
       'BuyerAgencyCompensation'],
      dtype='str')


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# =========================
# 0️⃣ Duplicate 
# =========================
total_duplicates = df.duplicated().sum()

if "ListingId" in df.columns:
    listing_duplicates = df.duplicated(subset=["ListingId"]).sum()
else:
    listing_duplicates = None

print("=== DUPLICATE CHECK ===")
print("Total duplicate rows:", total_duplicates)
print("Duplicate ListingId:", listing_duplicates)


# =========================
num_keep_cols = [
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
    'DaysOnMarket', 'YearBuilt',
    'LotSizeSquareFeet',
    'Latitude', 'Longitude',
    'ParkingTotal', 'GarageSpaces',
    'Stories',
    'month'
]

# 
num_keep_cols = [
    col for col in num_keep_cols
    if col in df.columns and df[col].isna().mean() < 1
]


# =========================
# 
# =========================
num_cols = df.select_dtypes(include=np.number).columns.tolist()
summary = df[num_cols].describe().T

report = []

for col in num_cols:
    col_data = df[col].dropna()

    missing_count = df[col].isna().sum()
    missing_pct = round(df[col].isna().mean() * 100, 2)

    if col_data.empty:
        report.append([col, False, missing_count, missing_pct, 0, 0,
                       np.nan, np.nan, np.nan, np.nan,
                       np.nan, np.nan, 0, False, False, "Drop"])
        continue

    # 
    min_val = summary.loc[col, 'min']
    max_val = summary.loc[col, 'max']
    std_val = summary.loc[col, 'std']
    mean_val = summary.loc[col, 'mean']

    # 
    negative_count = int((col_data < 0).sum())
    zero_count = int((col_data == 0).sum())

    # extreme max
    p99 = col_data.quantile(0.99)
    extreme_max = max_val > p99 * 5 if pd.notna(p99) and p99 != 0 else False

    # high std
    high_std = std_val > abs(mean_val) * 2 if pd.notna(mean_val) and mean_val != 0 else False

    # IQR
    q1 = col_data.quantile(0.25)
    q3 = col_data.quantile(0.75)
    iqr = q3 - q1

    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = int(((col_data < lower) | (col_data > upper)).sum())
    else:
        outlier_count = 0

    # 
    is_core = col in num_keep_cols

    report.append([
        col, is_core,
        missing_count, missing_pct,
        negative_count, zero_count,
        round(min_val, 2), round(max_val, 2),
        round(mean_val, 2), round(std_val, 2),
        round(q1, 2), round(q3, 2),
        outlier_count, extreme_max, high_std,
        None
    ])


# =========================
# 3️⃣ DataFrame
# =========================
report_df = pd.DataFrame(report, columns=[
    'Column', 'Is Core',
    'Missing', 'Missing %',
    'Negative Count', 'Zero Count',
    'Min', 'Max', 'Mean', 'Std',
    'Q1', 'Q3',
    'IQR Outliers',
    'Extreme Max ⚠️',
    'High Std ⚠️',
    'Action'
])


# =========================
# 4️⃣ Action ）
# =========================
def assign_action(row):
    if row["Missing %"] > 90:
        return "Drop"

    if row["Is Core"]:
        if row["Negative Count"] > 0:
            return "Fix Negative"
        elif row["Extreme Max ⚠️"] or row["High Std ⚠️"] or row["IQR Outliers"] > 0:
            return "Cap / Clean"
        else:
            return "OK"
    else:
        if row["Negative Count"] > 0:
            return "Fix Negative"
        elif row["Extreme Max ⚠️"] or row["High Std ⚠️"] or row["IQR Outliers"] > 0:
            return "Cap / Clean"
        else:
            return "OK"

report_df["Action"] = report_df.apply(assign_action, axis=1)


# =========================
# 5️⃣ ）
# =========================
action_order = {
    "Drop": 0,
    "Cap / Clean": 1,
    "Fix Negative": 2,
    "OK": 3
}

report_df["ActionRank"] = report_df["Action"].map(action_order)

report_df = report_df.sort_values(
    by=["ActionRank", "Missing %"],
    ascending=[True, False]
).reset_index(drop=True)


# =========================
# 6️⃣ 
# =========================


print("=== DROP ===")
display(report_df[report_df["Action"] == "Drop"])


print("=== CAP / CLEAN ===")
display(report_df[report_df["Action"] == "Cap / Clean"])


print("=== KEEP (OK + Fix Negative) ===")
display(report_df[report_df["Action"].isin(["OK", "Fix Negative"])])


print("=== KEEP (CORE) ===")
display(
    report_df[report_df["Is Core"]]
    .sort_values(by=["ActionRank", "Missing %"])
)

=== DUPLICATE CHECK ===
Total duplicate rows: 140
Duplicate ListingId: 574
=== DROP ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
0,FireplacesTotal,False,591071,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
1,AboveGradeFinishedArea,False,591071,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
2,ElementarySchoolDistrict,False,591071,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
3,CoveredSpaces,False,591071,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
4,MiddleOrJuniorSchoolDistrict,False,591071,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
5,TaxYear,False,590704,99.94,0,0,2009.0,2026.0,2022.97,2.43,2023.0,2024.0,27,False,False,Drop,0
6,TaxAnnualAmount,False,588564,99.58,0,485,0.0,10254000.0,18472.50,210564.57,244.5,19425.0,114,True,True,Drop,0
7,BelowGradeFinishedArea,False,588481,99.56,0,2155,0.0,17424.0,106.56,799.83,0.0,0.0,0,True,True,Drop,0


=== CAP / CLEAN ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
8,BuyerAgencyCompensation,False,523291,88.53,0,267,0.0,1.000000e+05,1.944100e+02,2152.72,2.000000e+00,2.500000e+00,8479,True,True,Cap / Clean,1
9,BuildingAreaTotal,False,514505,87.05,0,461,0.0,4.626070e+05,2.552360e+03,5192.31,1.161000e+03,2.651000e+03,6243,True,True,Cap / Clean,1
10,MainLevelBedrooms,False,277468,46.94,0,57567,0.0,3.333000e+03,1.930000e+00,6.25,1.000000e+00,3.000000e+00,498,True,True,Cap / Clean,1
11,AssociationFee,False,183224,31.00,0,232039,0.0,7.500000e+05,1.853600e+02,1633.63,0.000000e+00,2.950000e+02,17073,True,True,Cap / Clean,1
12,GarageSpaces,True,75663,12.80,0,82347,0.0,6.000000e+02,1.770000e+00,3.88,1.000000e+00,2.000000e+00,11202,True,True,Cap / Clean,1
13,LotSizeAcres,False,55951,9.47,0,12508,0.0,3.702600e+08,7.635900e+02,506344.08,1.200000e-01,3.000000e-01,84949,True,True,Cap / Clean,1
14,LotSizeSquareFeet,True,54540,9.23,0,12888,0.0,5.193920e+09,4.700943e+05,20781568.09,5.186000e+03,1.295800e+04,85007,True,True,Cap / Clean,1
15,LotSizeArea,False,54083,9.15,0,12911,0.0,9.187423e+08,4.811120e+04,2237845.08,4.941000e+03,1.148025e+04,82507,True,True,Cap / Clean,1
16,LivingArea,True,41721,7.06,0,484,0.0,9.090909e+08,3.497250e+03,1226756.09,1.183000e+03,2.150000e+03,25726,True,True,Cap / Clean,1
17,BedroomsTotal,True,39355,6.66,0,4583,0.0,1.230000e+02,3.080000e+00,1.40,2.000000e+00,4.000000e+00,2066,True,False,Cap / Clean,1


=== KEEP (OK + Fix Negative) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
26,Latitude,True,19015,3.22,5,43,-117.47,157.0,34.53,1.61,33.74,34.30,133038,False,False,Fix Negative,2
27,Longitude,True,18843,3.19,572123,43,-177.65,329.0,-118.45,3.11,-118.54,-117.33,100211,True,False,Fix Negative,2
28,ParkingTotal,True,18668,3.16,111,63198,-143.00,2593669.0,7.61,3439.17,2.00,3.00,107416,True,True,Fix Negative,2
29,DaysOnMarket,True,0,0.00,60,22147,-288.00,12430.0,43.28,69.83,9.00,54.00,43775,True,False,Fix Negative,2
30,Stories,True,118318,20.02,0,0,1.00,2.0,1.36,0.48,1.00,2.00,0,False,False,OK,3
31,month,True,0,0.00,0,0,202401.00,202603.0,202469.89,64.68,202407.00,202508.00,0,False,False,OK,3


=== KEEP (CORE) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
24,ClosePrice,True,7,0.00,0,35,0.00,9.895000e+08,874572.03,5275504.00,52500.00,1070000.00,27471,True,True,Cap / Clean,1
22,ListPrice,True,910,0.15,0,2,0.00,1.375000e+08,840013.43,1275659.68,62500.00,1050000.00,28195,True,False,Cap / Clean,1
20,OriginalListPrice,True,1711,0.29,0,6,0.00,1.390000e+09,906125.93,5745376.18,65000.00,1089000.00,27768,True,True,Cap / Clean,1
19,YearBuilt,True,24911,4.21,0,0,1776.00,2.026000e+03,1978.10,27.09,1960.00,1999.00,1523,False,False,Cap / Clean,1
18,BathroomsTotalInteger,True,27397,4.64,0,12463,0.00,1.750000e+02,2.45,1.42,2.00,3.00,37272,True,False,Cap / Clean,1
17,BedroomsTotal,True,39355,6.66,0,4583,0.00,1.230000e+02,3.08,1.40,2.00,4.00,2066,True,False,Cap / Clean,1
16,LivingArea,True,41721,7.06,0,484,0.00,9.090909e+08,3497.25,1226756.09,1183.00,2150.00,25726,True,True,Cap / Clean,1
14,LotSizeSquareFeet,True,54540,9.23,0,12888,0.00,5.193920e+09,470094.26,20781568.09,5186.00,12958.00,85007,True,True,Cap / Clean,1
12,GarageSpaces,True,75663,12.80,0,82347,0.00,6.000000e+02,1.77,3.88,1.00,2.00,11202,True,True,Cap / Clean,1
29,DaysOnMarket,True,0,0.00,60,22147,-288.00,1.243000e+04,43.28,69.83,9.00,54.00,43775,True,False,Fix Negative,2
